# Metropolis-Sampling im $\phi^4$-Potential

Wir berechnen den Erwartungswert

$$\langle O\rangle=\langle x^2\rangle,\qquad P(x)=Z^{-1}e^{-V(x)},$$

mit dem einwelligen, aber nichtlinearen Potential

$$V(x)=\frac{m^2}{2}x^2+\frac{\lambda}{4}x^4.$$

Wir wählen $m^2>0$. Dadurch besitzt die Verteilung nur ein Maximum und Ergodizitätsprobleme durch seltene Übergänge zwischen zwei Potentialmulden werden zunächst vermieden.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
m2 = 0.5
lam = 0.5

def potential(x):
    return 0.5 * m2 * x**2 + 0.25 * lam * x**4

# Referenzwert durch direkte eindimensionale Integration
x_grid = np.linspace(-7, 7, 200_001)
weight = np.exp(-potential(x_grid))
partition_function = np.trapezoid(weight, x_grid)
x2_exact = np.trapezoid(x_grid**2 * weight, x_grid) / partition_function

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(x_grid, potential(x_grid))
axes[0].set(xlim=(-3, 3), ylim=(0, 12), xlabel='$x$', ylabel='$V(x)$',
            title=r'Einwelliges $\phi^4$-Potential')
axes[1].plot(x_grid, weight / partition_function)
axes[1].set(xlim=(-3, 3), xlabel='$x$', ylabel='$P(x)$',
            title='Zielverteilung')
plt.tight_layout()
plt.show()
print(f'Referenzwert: <x^2> = {x2_exact:.6f}')

## Metropolis-Algorithmus

Ausgehend vom aktuellen Wert $x$ schlagen wir eine lokale Verschiebung

$$x'=x+\delta,\qquad \delta\sim\mathcal U(-\Delta,\Delta),$$

vor. Der Vorschlag wird mit Wahrscheinlichkeit $\min[1,e^{-\Delta V}]$ akzeptiert. Kleine $\Delta$ führen zu häufig akzeptierten, aber stark korrelierten Schritten. Bei sehr großen $\Delta$ werden dagegen viele Vorschläge verworfen.

In [ ]:
def metropolis(n_samples, proposal_width, warmup=5000, x0=0.0, rng=None):
    if rng is None:
        rng = np.random.default_rng()

    x = x0
    samples = np.empty(n_samples)
    accepted = 0
    total_steps = warmup + n_samples

    for step in range(total_steps):
        x_new = x + rng.uniform(-proposal_width, proposal_width)
        delta_v = potential(x_new) - potential(x)
        if delta_v <= 0 or rng.random() < np.exp(-delta_v):
            x = x_new
            if step >= warmup:
                accepted += 1
        if step >= warmup:
            samples[step - warmup] = x

    return samples, accepted / n_samples

proposal_width = 0.8
x_samples, acceptance = metropolis(100_000, proposal_width, rng=rng)
observable = x_samples**2

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(observable[:1500], linewidth=0.7)
axes[0].axhline(x2_exact, color='black', linestyle='--', label='Referenzwert')
axes[0].set(xlabel='Monte-Carlo-Schritt', ylabel='$O=x^2$', title='Zeitreihe')
axes[0].legend()
axes[1].hist(x_samples, bins=70, density=True, alpha=0.65, label='Metropolis')
axes[1].plot(x_grid, weight / partition_function, color='black', label='exakt')
axes[1].set(xlim=(-3, 3), xlabel='$x$', ylabel='Wahrscheinlichkeitsdichte',
            title=fr'$\Delta={proposal_width}$, Akzeptanz={acceptance:.2f}')
axes[1].legend()
plt.tight_layout()
plt.show()

## Naive Fehlerabschätzung

Wären die Messwerte $O_i=x_i^2$ unabhängig, hätte ihr Mittelwert den Standardfehler

$$\sigma_\mathrm{naiv}=\sqrt{\frac{\operatorname{Var}(O)}{N}}.$$

Die folgende Grafik zeigt den laufenden Mittelwert mit einem Band von $\pm3\sigma_\mathrm{naiv}$. Beim Metropolis-Verfahren sind aufeinanderfolgende Zustände jedoch korreliert. Das Band ist deshalb im Allgemeinen zu schmal.

In [ ]:
sample_counts = np.unique(np.geomspace(100, len(observable), 180).astype(int))
running_mean = np.array([observable[:n].mean() for n in sample_counts])
running_var = np.array([observable[:n].var(ddof=1) for n in sample_counts])
naive_error = np.sqrt(running_var / sample_counts)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sample_counts, running_mean, label='laufender Mittelwert')
ax.fill_between(sample_counts, running_mean - 3 * naive_error,
                running_mean + 3 * naive_error, alpha=0.25,
                label=r'naiv: $\pm3\sigma$')
ax.axhline(x2_exact, color='black', linestyle='--', label='Referenzwert')
ax.set_xscale('log')
ax.set(xlabel='Anzahl $N$ der Messungen', ylabel=r'$\overline{x^2}_N$',
       title='Naive Fehlerabschätzung')
ax.legend()
plt.tight_layout()
plt.show()
print(f'<x^2> = {observable.mean():.6f} ± {naive_error[-1]:.6f} (naiv, 1 sigma)')

## Autokorrelation und Vorschlagsbreite

Die normierte Autokorrelationsfunktion des Observablen $O$ ist

$$\rho(t)=\frac{\langle(O_i-\langle O\rangle)(O_{i+t}-\langle O\rangle)\rangle}{\operatorname{Var}(O)}.$$

Wir vergleichen mehrere Vorschlagsbreiten $\Delta$. An den anfänglichen positiven Zerfall passen wir das Modell $\rho(t)=e^{-t/\tau_\mathrm{exp}}$. Der Fit wird im halblogarithmischen Diagramm als Gerade ausgeführt; seine Steigung liefert $\tau_\mathrm{exp}$.

In [ ]:
def autocorrelation(data, max_lag):
    centered = np.asarray(data) - np.mean(data)
    n = len(centered)
    n_fft = 1 << (2 * n - 1).bit_length()
    spectrum = np.fft.rfft(centered, n=n_fft)
    corr = np.fft.irfft(spectrum * np.conjugate(spectrum), n=n_fft)[:n]
    corr /= np.arange(n, 0, -1)
    return corr[:max_lag + 1] / corr[0]

def fit_exponential_correlation(rho, min_rho=0.02):
    lags = np.arange(len(rho))
    mask = (lags > 0) & (rho > min_rho)
    # Nur der zusammenhängende anfängliche Zerfall wird verwendet.
    invalid = np.flatnonzero((lags > 0) & ~mask)
    stop = invalid[0] if len(invalid) else len(rho)
    fit_lags = lags[1:stop]
    fit_values = rho[1:stop]
    if len(fit_lags) < 3:
        raise RuntimeError('Zu wenige Punkte für den Autokorrelationsfit.')
    # Fit mit rho(0)=1, also log(rho)=-t/tau.
    slope = np.dot(fit_lags, np.log(fit_values)) / np.dot(fit_lags, fit_lags)
    return -1 / slope, fit_lags[-1]

proposal_widths = [0.15, 0.8, 3.0]
series = {}
fit_results = {}
max_lag = 500
Nsamples=1000000


fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for width in proposal_widths:
    samples, acc = metropolis(Nsamples, width, rng=rng)
    values = samples**2
    rho = autocorrelation(values, max_lag)
    tau_exp, fit_stop = fit_exponential_correlation(rho)
    series[width] = values
    fit_results[width] = (rho, tau_exp, acc)

    lags = np.arange(max_lag + 1)
    line = axes[0].plot(lags, rho, label=fr'$\Delta={width}$, Akz.={acc:.2f}')[0]
    fit_lags = np.arange(fit_stop + 1)
    axes[0].plot(fit_lags, np.exp(-fit_lags / tau_exp), '--',
                 color=line.get_color())
    positive = rho > 0
    axes[1].semilogy(lags[positive], rho[positive], color=line.get_color(),
                    label=fr'$\Delta={width}$, $\tau_\mathrm{{exp}}={tau_exp:.1f}$')
    axes[1].semilogy(fit_lags, np.exp(-fit_lags / tau_exp), '--',
                    color=line.get_color())

axes[0].set(xlabel='Abstand $t$', ylabel=r'$\rho(t)$',
            title='Autokorrelation von $O=x^2$', ylim=(-0.05, 1.02))
axes[1].set(xlabel='Abstand $t$', ylabel=r'$\rho(t)$',
            title='Exponentieller Fit', ylim=(1e-3, 1.05))
for ax in axes:
    ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

## Fehlerabschätzung mit Autokorrelationszeit

Für korrelierte Daten gilt

$$\operatorname{Var}(\bar O)=\frac{\operatorname{Var}(O)}{N}\left[1+2\sum_{t=1}^{\infty}\rho(t)\right]=\frac{g\,\operatorname{Var}(O)}{N}.$$

Der Faktor $g$ vergrößert den Fehler und reduziert die effektive Stichprobengröße auf $N_\mathrm{eff}=N/g$. Für den exponentiellen Fit ist mit $q=e^{-1/\tau_\mathrm{exp}}$

$$g=1+2\sum_{t=1}^{\infty}q^t=\frac{1+q}{1-q}.$$

Damit wird aus dem naiven Standardfehler $\sigma_\mathrm{korr}=\sqrt{g}\,\sigma_\mathrm{naiv}$.

In [ ]:
selected_width = 0.8
observable = series[selected_width]
rho, tau_exp, acceptance = fit_results[selected_width]
q = np.exp(-1 / tau_exp)
correlation_factor = (1 + q) / (1 - q)

sample_counts = np.unique(np.geomspace(100, len(observable), 180).astype(int))
running_mean = np.array([observable[:n].mean() for n in sample_counts])
running_var = np.array([observable[:n].var(ddof=1) for n in sample_counts])
naive_error = np.sqrt(running_var / sample_counts)
corrected_error = np.sqrt(correlation_factor) * naive_error

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sample_counts, running_mean, label='laufender Mittelwert')
ax.fill_between(sample_counts, running_mean - 3 * corrected_error,
                running_mean + 3 * corrected_error, alpha=0.25,
                label=r'korrigiert: $\pm3\sigma$')
ax.axhline(x2_exact, color='black', linestyle='--', label='Referenzwert')
ax.set_xscale('log')
ax.set(xlabel='Anzahl $N$ der Messungen', ylabel=r'$\overline{x^2}_N$',
       title=fr'Fehler mit Autokorrelation ($\Delta={selected_width}$)')
ax.legend()
plt.tight_layout()
plt.show()

print(f'tau_exp = {tau_exp:.2f}')
print(f'Korrelationsfaktor g = {correlation_factor:.2f}')
print(f'Effektive Stichprobengröße = {len(observable) / correlation_factor:.0f}')
print(f'<x^2> = {observable.mean():.6f} ± {corrected_error[-1]:.6f} (korrigiert, 1 sigma)')

## Aufgabe: Heuristische Binning-Analyse

Bestimmen Sie den Fehler von $\langle O\rangle$ direkt aus der korrelierten Datenreihe `observable`:

1. Teilen Sie die Zeitreihe für verschiedene Blockgrößen $M$ in $d=\lfloor N/M\rfloor$ vollständige Blöcke.
2. Berechnen Sie für jeden Block den Blockmittelwert $\bar O_M^{(j)}$.
3. Bestimmen Sie die Varianz der $d$ Blockmittelwerte und daraus

   $$\sigma_\mathrm{bin}(M)=\sqrt{\frac{\operatorname{Var}(\bar O_M)}{d}}.$$

4. Verwenden Sie Zweierpotenzen für $M$ und behalten Sie bei der größten Blockgröße noch mindestens 16 Blöcke. Plotten Sie $\sigma_\mathrm{bin}(M)$ gegen $M$.
5. Plotten Sie zusätzlich

   $$g(M)=\frac{M\operatorname{Var}(\bar O_M)}{\operatorname{Var}(O)}.$$

Für hinreichend große Datenmengen und Blockgrößen sollte ein Plateau entstehen. Warum? Wie liefert das Plateau den tatsächlichen Fehler von $\bar O$? Wie hängen der Plateauwert, die Autokorrelationszeit und die effektive Zahl unabhängiger Messungen zusammen? Warum werden die Ergebnisse für sehr große $M$ wieder unruhig?

In [ ]:
def binning_statistics(data, block_size):
    n_blocks = len(data) // block_size
    trimmed = data[:n_blocks * block_size]
    block_means = trimmed.reshape(n_blocks, block_size).mean(axis=1)
    error_of_mean = np.sqrt(block_means.var(ddof=1) / n_blocks)
    g_estimate = block_size * block_means.var(ddof=1) / np.var(data, ddof=1)
    return error_of_mean, g_estimate, n_blocks

min_blocks = 16
largest_block = max(1, len(observable) // min_blocks)
max_block_power = int(np.floor(np.log2(largest_block)))
block_sizes = 2 ** np.arange(max_block_power + 1)
bin_errors = []
g_binning = []
n_blocks = []
for block_size in block_sizes:
    error, g_estimate, count = binning_statistics(observable, block_size)
    bin_errors.append(error)
    g_binning.append(g_estimate)
    n_blocks.append(count)

bin_errors = np.asarray(bin_errors)
g_binning = np.asarray(g_binning)
naive_full_error = np.sqrt(np.var(observable, ddof=1) / len(observable))
fit_error = np.sqrt(correlation_factor) * naive_full_error

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(block_sizes, bin_errors, 'o-', label='Binning')
axes[0].axhline(naive_full_error, color='tab:red', linestyle=':', label='naiv')
axes[0].axhline(fit_error, color='black', linestyle='--', label='Autokorrelationsfit')
axes[0].set(xscale='log', xlabel='Blockgröße $M$',
            ylabel='Standardfehler des Mittelwerts', title='Binning-Fehler')
axes[0].legend()

axes[1].plot(block_sizes, g_binning, 'o-', label='$g(M)$ aus Binning')
axes[1].axhline(correlation_factor, color='black', linestyle='--',
                label='$g$ aus Autokorrelationsfit')
axes[1].set(xscale='log', xlabel='Blockgröße $M$', ylabel='$g(M)$',
            title='Suche nach einem Plateau')
axes[1].legend()
plt.tight_layout()
plt.show()

print('Blockgröße    Zahl der Blöcke    Fehler')
for size, count, error in zip(block_sizes, n_blocks, bin_errors):
    print(f'{size:10d}    {count:15d}    {error:.6f}')

### Kurze Auswertung

Für $M\gg\tau_\mathrm{int}$ sind benachbarte Blockmittelwerte näherungsweise unabhängig. Deshalb erreichen $\sigma_\mathrm{bin}(M)$ und $g(M)$ ein Plateau. Das Plateau von $\sigma_\mathrm{bin}$ ist bereits der Standardfehler $\epsilon$ des Gesamtmittelwerts. 

Vergleich mit dem Ausdruck 

$$\epsilon=\Delta O\sqrt{\frac{2\tau_\mathrm{int}}{N}}$$

für den Fehler in Abhängigkeit der Autokorrelationszeit ergibt sich am Plateau $g(M)=2\tau_\mathrm{int}$. 


Bei sehr großen Blöcken bleiben zu wenige Blockmittelwerte für eine stabile Varianzschätzung.